# BIM Quality Checker Demo

This notebook demonstrates **bim-quality-checker**, a library that evaluates BIM model quality by comparing geometry against point cloud measurements (e.g. laser scans).

## What it computes

### Geometric metrics
| Metric | Description |
|---|---|
| **Accuracy** | Fraction of BIM surface samples within a distance threshold of the scan. High accuracy means the model has no spurious geometry. |
| **Completeness** | Fraction of scan points within a distance threshold of the BIM surface. High completeness means the model captures most of the real structure. |
| **Chamfer Distance** | Symmetric mean nearest-neighbour distance between two point sets (lower is better). |
| **Coverage Ratio** | Ratio of the point cloud that is covered by the BIM surface. |

### Topological checks
| Check | Description |
|---|---|
| **Wall Connectivity** | Connected-component analysis to detect gaps between wall surfaces. |
| **Room Closure** | Watertightness and non-manifold edge tests to verify enclosed rooms. |

### Grading system
The average of accuracy and completeness yields a letter grade:
- **A** (>= 95%) -- Excellent
- **B** (>= 85%) -- Good
- **C** (>= 70%) -- Acceptable
- **D** (< 70%) -- Needs improvement

### Works with any IFC
The library loads IFC files via `ifcopenshell`, extracts all building-element surfaces, and evaluates them against any point cloud format supported by Open3D (PLY, PCD, XYZ, etc.).

In this demo we skip IFC loading and work directly with Open3D meshes to keep things self-contained.

## 1. Setup

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from bim_quality_checker.metrics import (
    accuracy,
    completeness,
    chamfer_distance,
    coverage_ratio,
    evaluate_all,
    check_wall_connectivity,
    check_room_closure,
    point_to_surface_distances,
)
from bim_quality_checker.report import generate_report
from bim_quality_checker.deviation_map import compute_deviation_map, deviation_colors
from bim_quality_checker.element_checker import ElementChecker

## 2. Create synthetic BIM model and point cloud

We create a simple **cube mesh** to act as a BIM model, and sample a **point cloud on its surface** to simulate a laser scan. A small amount of Gaussian noise is added to the scan to make it realistic.

In [ ]:
# Create a cube mesh (our "BIM model")
mesh = o3d.geometry.TriangleMesh.create_box(width=2.0, height=3.0, depth=2.0)
mesh.translate([-1.0, -1.5, -1.0])  # centre at origin
mesh.compute_vertex_normals()

print(f"BIM mesh: {len(np.asarray(mesh.vertices))} vertices, "
      f"{len(np.asarray(mesh.triangles))} triangles")

# Sample a point cloud on the mesh surface (our "scan")
n_scan_points = 10_000
pcd = mesh.sample_points_uniformly(number_of_points=n_scan_points)

# Add realistic measurement noise (sigma = 5 mm)
noise = np.random.default_rng(42).normal(0, 0.005, size=(n_scan_points, 3))
pcd.points = o3d.utility.Vector3dVector(np.asarray(pcd.points) + noise)

print(f"Scan point cloud: {len(np.asarray(pcd.points))} points")
print(f"Noise level: sigma = 5 mm")

## 3. Compute geometric metrics

We compute the four core metrics that quantify how well the BIM model matches the scan.

In [ ]:
threshold = 0.05  # 50 mm tolerance

# --- Accuracy: sample BIM surface, measure distance to scan ---
n_mesh_samples = 50_000
mesh_samples = mesh.sample_points_uniformly(number_of_points=n_mesh_samples)
mesh_pts = np.asarray(mesh_samples.points)

pcd_tree = o3d.geometry.KDTreeFlann(pcd)
acc_dists = []
for pt in mesh_pts:
    _, _, d2 = pcd_tree.search_knn_vector_3d(pt, 1)
    acc_dists.append(np.sqrt(d2[0]))
acc_dists = np.array(acc_dists)

acc = accuracy(acc_dists, threshold)
print(f"Accuracy  : {acc * 100:.1f}%")

# --- Completeness: measure scan-to-surface distance ---
pcd_pts = np.asarray(pcd.points)
dists_pcd_to_mesh = point_to_surface_distances(pcd_pts, mesh)

comp = completeness(dists_pcd_to_mesh, threshold)
print(f"Completeness : {comp * 100:.1f}%")

# --- Chamfer distance ---
cd = chamfer_distance(pcd_pts, mesh_pts)
print(f"Chamfer distance : {cd:.6f} m^2")

# --- Coverage ratio ---
cov = coverage_ratio(pcd_pts, mesh, threshold)
print(f"Coverage ratio : {cov * 100:.1f}%")

## 4. Topology checks

These heuristic checks verify structural properties of the BIM mesh.

In [ ]:
# Wall connectivity: is the mesh a single connected component?
wall_result = check_wall_connectivity(mesh)
print("Wall Connectivity")
print(f"  Components       : {wall_result['n_components']}")
print(f"  Single component : {wall_result['is_single_component']}")
print(f"  Largest component: {wall_result['largest_component_triangles']} triangles")
print()

# Room closure: is the mesh watertight (enclosed rooms)?
room_result = check_room_closure(mesh)
print("Room Closure")
print(f"  Watertight         : {room_result['is_watertight']}")
print(f"  Non-manifold edges : {room_result['n_non_manifold_edges']}")

## 5. Generate text report with grade

The `evaluate_all()` function runs all metrics at once, and `generate_report()` formats the results as a graded report.

In [ ]:
# Run the full evaluation pipeline
results = evaluate_all(mesh, pcd, distance_threshold=threshold, n_mesh_samples=50_000)

# Generate a text report
report_text = generate_report(results, fmt="text")
print(report_text)

In [ ]:
# The report is also available in Markdown
from IPython.display import Markdown

report_md = generate_report(results, fmt="markdown")
Markdown(report_md)

## 6. Deviation map

The deviation map computes per-point distances from the scan to the BIM surface and visualises them with a blue-green-red colour ramp (blue = close, red = far).

In [ ]:
# Compute per-point deviations
deviations = compute_deviation_map(mesh, pcd)

print(f"Deviation statistics:")
print(f"  Mean   : {np.mean(deviations)*1000:.2f} mm")
print(f"  Median : {np.median(deviations)*1000:.2f} mm")
print(f"  Std    : {np.std(deviations)*1000:.2f} mm")
print(f"  Max    : {np.max(deviations)*1000:.2f} mm")
print(f"  95th %%: {np.percentile(deviations, 95)*1000:.2f} mm")

In [ ]:
# Colour-code the point cloud by deviation
colors = deviation_colors(deviations, vmin=0.0, vmax=0.02)  # 0-20 mm range

# Histogram of deviations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram
axes[0].hist(deviations * 1000, bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(threshold * 1000, color="red", linestyle="--", label=f"Threshold ({threshold*1000:.0f} mm)")
axes[0].set_xlabel("Deviation (mm)")
axes[0].set_ylabel("Point count")
axes[0].set_title("Distribution of point-to-surface deviations")
axes[0].legend()

# Right: scatter plot of points coloured by deviation (XY projection)
pts = np.asarray(pcd.points)
sc = axes[1].scatter(pts[:, 0], pts[:, 1], c=colors, s=0.5, alpha=0.8)
axes[1].set_xlabel("X (m)")
axes[1].set_ylabel("Y (m)")
axes[1].set_title("Deviation map (XY projection)")
axes[1].set_aspect("equal")

# Add a custom colourbar
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

cmap_colors = [(0, 0, 1), (0, 1, 0), (1, 0, 0)]  # blue -> green -> red
cmap = mcolors.LinearSegmentedColormap.from_list("deviation", cmap_colors)
norm = mcolors.Normalize(vmin=0, vmax=20)
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=axes[1])
cbar.set_label("Deviation (mm)")

plt.tight_layout()
plt.show()

In [ ]:
# 3D scatter coloured by deviation (XZ and YZ views)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(pts[:, 0], pts[:, 2], c=colors, s=0.5, alpha=0.8)
axes[0].set_xlabel("X (m)")
axes[0].set_ylabel("Z (m)")
axes[0].set_title("Deviation map (XZ projection)")
axes[0].set_aspect("equal")

axes[1].scatter(pts[:, 1], pts[:, 2], c=colors, s=0.5, alpha=0.8)
axes[1].set_xlabel("Y (m)")
axes[1].set_ylabel("Z (m)")
axes[1].set_title("Deviation map (YZ projection)")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

## 7. Per-element analysis concept

In a real IFC workflow, the `ElementChecker` extracts each building element (IfcWall, IfcSlab, IfcDoor, etc.) and evaluates it individually. Here we simulate this by splitting our cube into separate face groups and evaluating each one.

In [ ]:
# Simulate per-element analysis with separate meshes for each face of the cube
checker = ElementChecker(distance_threshold=threshold, n_mesh_samples=2000)

# Define 6 "elements" (the faces of the cube)
element_defs = [
    {"name": "Floor",     "type": "IfcSlab",   "center": [ 0.0, -1.5,  0.0], "normal": [0, -1, 0]},
    {"name": "Ceiling",   "type": "IfcSlab",   "center": [ 0.0,  1.5,  0.0], "normal": [0,  1, 0]},
    {"name": "North Wall","type": "IfcWall",   "center": [ 0.0,  0.0, -1.0], "normal": [0, 0, -1]},
    {"name": "South Wall","type": "IfcWall",   "center": [ 0.0,  0.0,  1.0], "normal": [0, 0,  1]},
    {"name": "East Wall", "type": "IfcWall",   "center": [ 1.0,  0.0,  0.0], "normal": [1, 0,  0]},
    {"name": "West Wall", "type": "IfcWall",   "center": [-1.0,  0.0,  0.0], "normal": [-1, 0, 0]},
]

print(f"{'Element':<15} {'Type':<10} {'Accuracy':>10} {'Completeness':>14} {'Mean Dev':>10} {'Pass':>6}")
print("-" * 70)

for i, edef in enumerate(element_defs):
    # Create a thin box to represent each face
    n = np.array(edef["normal"], dtype=float)
    c = np.array(edef["center"], dtype=float)
    abs_n = np.abs(n)

    # Size: large in tangent directions, thin in normal direction
    size = np.where(abs_n > 0.5, 0.01, [2.0, 3.0, 2.0])
    elem_mesh = o3d.geometry.TriangleMesh.create_box(
        width=float(size[0]), height=float(size[1]), depth=float(size[2])
    )
    elem_mesh.translate(c - size / 2)
    elem_mesh.compute_vertex_normals()

    result = checker.check_element(
        elem_mesh, pcd,
        element_type=edef["type"],
        global_id=f"GUID-{i:04d}",
        name=edef["name"],
    )

    status = "PASS" if result.passed else "FAIL"
    print(f"{result.name:<15} {result.ifc_type:<10} "
          f"{result.accuracy*100:>9.1f}% "
          f"{result.completeness*100:>13.1f}% "
          f"{result.mean_deviation*1000:>8.2f} mm "
          f"{status:>6}")

In [ ]:
# In production, the full pipeline would look like:
#
#   from bim_quality_checker.loader import load_ifc_surfaces, load_point_cloud
#   from bim_quality_checker.element_checker import ElementChecker
#
#   mesh = load_ifc_surfaces("building.ifc")
#   pcd = load_point_cloud("scan.ply")
#
#   # Global evaluation
#   results = evaluate_all(mesh, pcd)
#   results["element_breakdown"] = ElementChecker().check_ifc("building.ifc", pcd).to_dict()
#   print(generate_report(results))
#
#   # Deviation map export
#   from bim_quality_checker.deviation_map import export_colored_pointcloud, compute_deviation_map
#   devs = compute_deviation_map(mesh, pcd)
#   export_colored_pointcloud(pcd, devs, "deviation_map.ply")

print("Done! See the modules in bim_quality_checker for full API details.")